<h1>Imports</h1>

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

C:\Users\MSH\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h1>Load Data</h1>

In [2]:
file_path = "../../Data/Preprocessed/"

data = pd.read_parquet(file_path + "featured_training_data_V2.parquet")
print("Shape:", data.shape)
print("Sales range:", data['Sales'].min().round(2), "→", data['Sales'].max().round(2))

Shape: (844338, 19)
Sales range: 3.83 → 10.63


<h1>Train/Test Split <span style="font-size: 22px;">By date not random</span></h1>


In [3]:
FEATURES = [
    'DayOfWeek', 'CompetitionDistance', 'CompetitionDistanceMissing',
    'CompetitionOpenMissing', 'StateHoliday', 'SchoolHoliday',
    'Promo', 'Promo2', 'StoreType', 'Assortment',
    'Year', 'Week', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd',
    'CompetitionOpenMonths', 'Promo2ActiveWeeks', 'IsPromo2Active'
]
TARGET = 'Sales'

X = data[FEATURES]
y = data[TARGET]

SPLIT_RATIO = 0.85
split_idx = int(len(data) * SPLIT_RATIO)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (717687, 18) | Test: (126651, 18)


<h1>Evaluation Function</h1>

In [4]:
def evaluate(y_true, y_pred, model_name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    mae_orig  = mean_absolute_error(np.expm1(y_true), np.expm1(y_pred))
    rmse_orig = np.sqrt(mean_squared_error(np.expm1(y_true), np.expm1(y_pred)))

    print(f"\n{'='*40}")
    print(f"  {model_name}")
    print(f"{'='*40}")
    print(f"  [Log Scale]")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²:   {r2:.4f}")
    print(f"\n  [Original Scale (Sales in €)]")
    print(f"  MAE:  {mae_orig:.2f}")
    print(f"  RMSE: {rmse_orig:.2f}")

    return {"mae": mae, "rmse": rmse, "r2": r2,
            "mae_orig": mae_orig, "rmse_orig": rmse_orig}